# 18xD — Trading robustness and dissertation figures

This stage reports cost and threshold sensitivity, date-level bootstrap uncertainty and thesis-ready trading figures. The primary conclusion is preserved even when unfavourable: the development-frozen empirical strategy is profitable on the small internal holdout but loses on the June external block.

**Revision v2.** Headline drawdowns use a zero initial portfolio value, and both cumulative-PnL figures display an explicit zero starting point.

In [1]:
from __future__ import annotations
import hashlib, json, math, platform, sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
ROOT=Path.cwd().resolve()
if not (ROOT/'.git').exists(): raise RuntimeError(f'Run from repository root, not {ROOT}')
UTC=timezone.utc

def sha(path:Path)->str:
    h=hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
    return h.hexdigest()

def parse_bool(s:pd.Series,name:str)->pd.Series:
    if pd.api.types.is_bool_dtype(s): return s.astype(bool)
    out=s.astype(str).str.strip().str.lower().map({'true':True,'false':False,'1':True,'0':False,'yes':True,'no':False})
    if out.isna().any(): raise ValueError(f'Cannot parse Boolean {name}: {s[out.isna()].drop_duplicates().tolist()}')
    return out.astype(bool)

def verify_manifest(path:Path):
    m=pd.read_csv(path); failures=[]
    for r in m.itertuples(index=False):
        p=ROOT/r.path
        if not p.is_file(): failures.append(f'MISSING {r.path}'); continue
        if sha(p)!=r.sha256: failures.append(f'HASH {r.path}')
        if p.stat().st_size!=int(r.size_bytes): failures.append(f'SIZE {r.path}')
    if failures: raise AssertionError(f'Manifest failed {path}:\\n'+'\\n'.join(failures))

def write_manifest(out:Path, report_dir:Path, filename:str):
    rows=[]
    for root in [out,report_dir]:
        for p in sorted(root.rglob('*')):
            if p.is_file() and p.name!=filename:
                rows.append({'path':str(p.relative_to(ROOT)),'size_bytes':p.stat().st_size,'sha256':sha(p)})
    pd.DataFrame(rows).to_csv(out/filename,index=False)

def save_frame(frame:pd.DataFrame,path:Path):
    x=frame.copy()
    for c in x.columns:
        if pd.api.types.is_datetime64_any_dtype(x[c]):
            if getattr(x[c].dt,'tz',None) is not None: x[c]=x[c].astype('string')
            else: x[c]=x[c].dt.strftime('%Y-%m-%d')
    x.to_csv(path,index=False)

def max_drawdown(daily:pd.Series)->float:
    if daily.empty: return float('nan')
    cumulative=daily.sort_index().cumsum().to_numpy(dtype=float)
    running_peak=np.maximum.accumulate(np.concatenate(([0.0],cumulative)))[1:]
    return float(np.min(cumulative-running_peak))

STEP='18xD'; import matplotlib.pyplot as plt
XA=ROOT/'data/processed/18xA_development_trading_selection'; XC=ROOT/'data/processed/18xC_controlled_unblinding_trading_simulation'
PERF=XA/'18xA_development_threshold_performance.csv'; REG=XA/'18xA_selected_strategy_registry.csv'; XA_SUM=XA/'18xA_summary.json'; XA_MAN=XA/'18xA_sha256_manifest.csv'
TRADE=XC/'18xC_cost_sensitivity_trade_panel.csv'; DAILY=XC/'18xC_daily_pnl_panel.csv'; SUMMARY=XC/'18xC_strategy_cost_summary.csv'; XC_SUM=XC/'18xC_summary.json'; XC_MAN=XC/'18xC_sha256_manifest.csv'
OUT=ROOT/'data/processed/18xD_trading_robustness_and_figures'; REPORT=ROOT/'reports/18xD_trading_robustness_and_figures'; FIG=REPORT/'figures'; OUT.mkdir(parents=True,exist_ok=True); FIG.mkdir(parents=True,exist_ok=True)
for p in [PERF,REG,XA_SUM,XA_MAN,TRADE,DAILY,SUMMARY,XC_SUM,XC_MAN]:
    if not p.is_file(): raise FileNotFoundError(p)
for p in [XA_MAN,XC_MAN]: verify_manifest(p)
if json.loads(XA_SUM.read_text()).get('verdict')!='PASS' or json.loads(XC_SUM.read_text()).get('verdict')!='PASS': raise AssertionError('Upstream not PASS')
perf=pd.read_csv(PERF); registry=pd.read_csv(REG); trade=pd.read_csv(TRADE,dtype={'market_id':str},low_memory=False); daily=pd.read_csv(DAILY); summary=pd.read_csv(SUMMARY)
for d in [trade,daily]: d['event_date']=pd.to_datetime(d.event_date,errors='raise')
# date-level non-parametric bootstrap, including zero-PnL opportunity dates
rng=np.random.default_rng(20260722); boot=[]
primary_daily=daily.loc[daily.cost_per_share.eq(0.01)].copy()
for keys,g in primary_daily.groupby(['strategy_role','candidate_id','evaluation_block'],sort=True):
    vals=g.sort_values('event_date').daily_net_pnl.to_numpy(float); n=len(vals); draws=rng.choice(vals,size=(5000,n),replace=True).mean(axis=1)
    boot.append({'strategy_role':keys[0],'candidate_id':keys[1],'evaluation_block':keys[2],'dates':n,'mean_daily_net_pnl':float(vals.mean()),'bootstrap_mean_daily_pnl_ci_lower':float(np.quantile(draws,0.025)),'bootstrap_mean_daily_pnl_ci_upper':float(np.quantile(draws,0.975)),'bootstrap_probability_mean_positive':float((draws>0).mean()),'bootstrap_replications':5000})
bootstrap=pd.DataFrame(boot)
# development robustness for selected candidate/rule/variant across thresholds and costs
selected_perf=perf.merge(registry[['strategy_role','candidate_id','probability_variant','decision_rule','threshold']].rename(columns={'threshold':'selected_threshold'}),on=['candidate_id','probability_variant','decision_rule'],how='inner',validate='many_to_one')
# figure helpers
labels={'PRIMARY_OVERALL':'Empirical','GAUSSIAN_PROCESS_FAMILY':'Matérn GP','TREE_FAMILY':'CatBoost'}
fig_rows=[]
def savefig(name,title):
    path=FIG/name; plt.title(title); plt.tight_layout(); plt.savefig(path,dpi=220,bbox_inches='tight'); plt.close(); fig_rows.append({'figure_file':str(path.relative_to(ROOT)),'title':title,'sha256':sha(path)})
# 1 development threshold curves at primary cost
plt.figure(figsize=(8,5))
for role,g in selected_perf.loc[selected_perf.cost_per_share.eq(0.01)].groupby('strategy_role'):
    plt.plot(g.threshold,g.total_net_pnl,marker='o',label=labels[role]); s=g.loc[np.isclose(g.threshold,g.selected_threshold)]; plt.scatter(s.threshold,s.total_net_pnl,s=80)
plt.axhline(0,linewidth=1); plt.xlabel('Edge threshold'); plt.ylabel('Development net PnL'); plt.legend(); savefig('18xD_development_threshold_sensitivity.png','Development threshold sensitivity at 0.01 cost')
# 2/3 cumulative pnl
for block,filename,title in [('INTERNAL_HOLDOUT','18xD_holdout_cumulative_pnl.png','Internal-holdout cumulative PnL'),('EXTERNAL_TEST','18xD_external_cumulative_pnl.png','June external cumulative PnL')]:
    plt.figure(figsize=(8,5)); q=primary_daily.loc[primary_daily.evaluation_block.eq(block)]
    for role,g in q.groupby('strategy_role'):
        g=g.sort_values('event_date'); start_date=g.event_date.min()-pd.Timedelta(days=1); plot_dates=np.concatenate(([np.datetime64(start_date)],g.event_date.to_numpy(dtype='datetime64[ns]'))); plot_pnl=np.concatenate(([0.0],g.cumulative_net_pnl.to_numpy(dtype=float))); plt.plot(plot_dates,plot_pnl,marker='o',label=labels[role])
    plt.axhline(0,linewidth=1); plt.xlabel('Settlement date'); plt.ylabel('Cumulative net PnL'); plt.legend(); plt.xticks(rotation=30); savefig(filename,title+' (zero initial value)')
# 4/5 cost sensitivity
for block,filename,title in [('INTERNAL_HOLDOUT','18xD_holdout_cost_sensitivity.png','Internal-holdout cost sensitivity'),('EXTERNAL_TEST','18xD_external_cost_sensitivity.png','June external cost sensitivity')]:
    plt.figure(figsize=(8,5)); q=summary.loc[summary.evaluation_block.eq(block)]
    for role,g in q.groupby('strategy_role'):
        plt.plot(g.cost_per_share,g.total_net_pnl,marker='o',label=labels[role])
    plt.axhline(0,linewidth=1); plt.xlabel('Hypothetical cost per share'); plt.ylabel('Total net PnL'); plt.legend(); savefig(filename,title)
# 6 headline primary cost
head=summary.loc[summary.cost_per_share.eq(0.01)].copy(); head['label']=head.strategy_role.map(labels)+' — '+head.evaluation_block.map({'INTERNAL_HOLDOUT':'Holdout','EXTERNAL_TEST':'June'})
plt.figure(figsize=(8,5)); plt.barh(head.label,head.total_net_pnl); plt.axvline(0,linewidth=1); plt.xlabel('Total net PnL at 0.01 cost'); savefig('18xD_primary_cost_headline.png','Frozen-strategy trading results')
# 7 return and trade count
plt.figure(figsize=(8,5)); x=np.arange(len(head)); plt.bar(x,head.return_on_capital); plt.xticks(x,head.label,rotation=35,ha='right'); plt.axhline(0,linewidth=1); plt.ylabel('Return on committed capital'); savefig('18xD_return_on_capital.png','Return on committed capital')
figures=pd.DataFrame(fig_rows)
# headline table
headline=head[['strategy_role','candidate_id','probability_variant','decision_rule','evaluation_block','opportunity_books','trade_count','total_net_pnl','return_on_capital','hit_rate','max_drawdown','break_even_cost_per_trade']].merge(bootstrap,on=['strategy_role','candidate_id','evaluation_block'],how='left',validate='one_to_one')
checks=pd.DataFrame([{'check':'bootstrap_rows_6','passed':len(bootstrap)==6,'detail':str(len(bootstrap)),'blocking':True},{'check':'headline_rows_6','passed':len(headline)==6,'detail':str(len(headline)),'blocking':True},{'check':'figures_7','passed':len(figures)==7,'detail':str(len(figures)),'blocking':True},{'check':'figure_files_exist','passed':all((ROOT/p).is_file() for p in figures.figure_file),'detail':'all figure files','blocking':True},{'check':'primary_external_negative','passed':float(headline.loc[headline.strategy_role.eq('PRIMARY_OVERALL')&headline.evaluation_block.eq('EXTERNAL_TEST'),'total_net_pnl'].iloc[0])<0,'detail':'out-of-time loss retained','blocking':True},{'check':'corrected_primary_drawdowns','passed':all(np.isclose(headline.loc[headline.strategy_role.eq(k[0])&headline.evaluation_block.eq(k[1]),'max_drawdown'].iloc[0],v) for k,v in {('PRIMARY_OVERALL','INTERNAL_HOLDOUT'):-0.5725,('PRIMARY_OVERALL','EXTERNAL_TEST'):-1.4925,('GAUSSIAN_PROCESS_FAMILY','INTERNAL_HOLDOUT'):-0.3730,('GAUSSIAN_PROCESS_FAMILY','EXTERNAL_TEST'):-1.1975,('TREE_FAMILY','INTERNAL_HOLDOUT'):-0.5505,('TREE_FAMILY','EXTERNAL_TEST'):-1.9275}.items()),'detail':'running peak includes initial zero','blocking':True}])
issues=pd.DataFrame(columns=['issue_level','issue_code','strategy_role','evaluation_block','detail','blocking'])
for name,frame in {'bootstrap_summary':bootstrap,'development_selected_strategy_robustness':selected_perf,'headline_trading_summary':headline,'figure_inventory':figures,'integrity_checks':checks,'issues':issues}.items(): save_frame(frame,OUT/f'18xD_{name}.csv')
protocol={'step':STEP,'generated_at_utc':datetime.now(UTC).isoformat(),'verdict':'PASS','bootstrap':'5000 date-level non-parametric replications of mean daily PnL, including no-trade opportunity dates','figures':7,'primary_result_interpretation':'descriptive; small holdout and externally negative primary strategy','execution_caveat':'observed YES price is a fill proxy and not a bid/ask quote','maximum_drawdown_definition':'minimum cumulative PnL relative to running peak including initial portfolio value zero','cumulative_pnl_figures_include_zero_start':True}
(OUT/'18xD_protocol.json').write_text(json.dumps(protocol,indent=2),encoding='utf-8')
summary_json={'step':STEP,'generated_at_utc':datetime.now(UTC).isoformat(),'verdict':'PASS','bootstrap_rows':6,'headline_rows':6,'figure_rows':7,'primary_holdout_net_pnl':float(headline.loc[headline.strategy_role.eq('PRIMARY_OVERALL')&headline.evaluation_block.eq('INTERNAL_HOLDOUT'),'total_net_pnl'].iloc[0]),'primary_external_net_pnl':float(headline.loc[headline.strategy_role.eq('PRIMARY_OVERALL')&headline.evaluation_block.eq('EXTERNAL_TEST'),'total_net_pnl'].iloc[0]),'issue_rows':0,'integrity_checks_passed':int(checks.passed.sum()),'integrity_checks_total':len(checks)}
(OUT/'18xD_summary.json').write_text(json.dumps(summary_json,indent=2),encoding='utf-8')
pd.DataFrame([{'input_role':'18xA_development_threshold_performance','path':str(PERF.relative_to(ROOT)),'rows':len(perf),'sha256':sha(PERF)},{'input_role':'18xA_selected_strategy_registry','path':str(REG.relative_to(ROOT)),'rows':len(registry),'sha256':sha(REG)},{'input_role':'18xC_trade_panel','path':str(TRADE.relative_to(ROOT)),'rows':len(trade),'sha256':sha(TRADE)},{'input_role':'18xC_daily_pnl','path':str(DAILY.relative_to(ROOT)),'rows':len(daily),'sha256':sha(DAILY)},{'input_role':'18xC_strategy_cost_summary','path':str(SUMMARY.relative_to(ROOT)),'rows':len(summary),'sha256':sha(SUMMARY)}]).to_csv(OUT/'18xD_source_inventory.csv',index=False)
(OUT/'18xD_environment.json').write_text(json.dumps({'generated_at_utc':datetime.now(UTC).isoformat(),'python':sys.version,'platform':platform.platform(),'pandas':pd.__version__,'numpy':np.__version__,'matplotlib':__import__('matplotlib').__version__,'revision':'v2'},indent=2),encoding='utf-8')
lines=['# 18xD trading robustness and figures','','**PASS**','','The primary empirical strategy earns 0.2625 units on the ten-date internal holdout at a 0.01 cost, but loses 1.4925 units in June. This sign reversal is the principal out-of-time trading conclusion. The GP family also reverses sign; the tree family is negative in both blocks.','','Bootstrap intervals are descriptive because the samples contain only ten and thirty settlement dates. The execution proxy omits bid-ask spreads, liquidity constraints, slippage, partial fills and market impact.']
(REPORT/'18xD_trading_robustness_and_figures_report.md').write_text('\n'.join(lines)+'\n',encoding='utf-8')
write_manifest(OUT,REPORT,'18xD_sha256_manifest.csv'); print(json.dumps(summary_json,indent=2)); print('18xD PASS')

{
  "step": "18xD",
  "generated_at_utc": "2026-07-22T15:21:51.157516+00:00",
  "verdict": "PASS",
  "bootstrap_rows": 6,
  "headline_rows": 6,
  "figure_rows": 7,
  "primary_holdout_net_pnl": 0.2624999999999999,
  "primary_external_net_pnl": -1.4925,
  "issue_rows": 0,
  "integrity_checks_passed": 6,
  "integrity_checks_total": 6
}
18xD PASS
